In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

import json
import pickle

import numpy as np
import pandas as pd
import torch

from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import mutual_info_regression

import itertools

from configs import config

In [2]:
# ============================================================
# REPOSITORY CONFIGURATION
# ============================================================
 
# Project Root
PROJECT_ROOT = config.PROJECT_ROOT

# Data Paths
RAW_DATA_PATH = config.RAW_DATA_FILE
PROCESSED_DATA_PATH = config.PROCESSED_DATA_FILE

# Artifact Directory
ARTIFACT_DIR = config.ARTIFACT_DIR
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

if not PROCESSED_DATA_PATH.exists():
    raise FileNotFoundError(f"Missing processed data file: {PROCESSED_DATA_PATH}")

In [3]:
df = pd.read_csv(
    PROCESSED_DATA_PATH,
    parse_dates=["timestamp"]
)

print(f"Dataset Shape : {df.shape}")

df.head()

Dataset Shape : (1065289, 8)


,timestamp,Active_Power,Weather_Temperature_Celsius,Weather_Relative_Humidity,Global_Horizontal_Radiation,Diffuse_Horizontal_Radiation,Radiation_Global_Tilted,Radiation_Diffuse_Tilted
0,2014-01-01 00:00:00,-0.725207,30.081875,12.679476,1.267926,0.587425,0.928665,1.656146
1,2014-01-01 00:05:00,-0.717201,29.945786,12.650104,2.656051,1.286431,1.383545,1.948417
2,2014-01-01 00:10:00,-0.694308,29.381653,12.909550,4.119638,2.647693,2.077302,2.563462
3,2014-01-01 00:15:00,-0.682230,28.800579,13.392023,4.079689,2.499754,2.314080,2.791505
4,2014-01-01 00:20:00,-0.676284,28.300385,13.731309,4.032659,2.465453,2.441365,2.923790


In [4]:
TARGET_COLUMN = "Active_Power"

FEATURE_COLUMNS = [
    "Weather_Temperature_Celsius",
    "Weather_Relative_Humidity",
    "Global_Horizontal_Radiation",
    "Diffuse_Horizontal_Radiation",
    "Radiation_Global_Tilted",
    "Radiation_Diffuse_Tilted",
    "Active_Power",
]

X = df[FEATURE_COLUMNS]
y = df[TARGET_COLUMN]

print("Feature Shape :", X.shape)
print("Target Shape  :", y.shape)

Feature Shape : (1065289, 7)
Target Shape  : (1065289,)


In [5]:
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15

n_samples = len(df)

train_end = int(TRAIN_RATIO * n_samples)
val_end = train_end + int(VAL_RATIO * n_samples)

X_train = X.iloc[:train_end]
X_val = X.iloc[train_end:val_end]
X_test = X.iloc[val_end:]

y_train = y.iloc[:train_end]
y_val = y.iloc[train_end:val_end]
y_test = y.iloc[val_end:]

print(f"Train Samples      : {len(X_train)}")
print(f"Validation Samples : {len(X_val)}")
print(f"Test Samples       : {len(X_test)}")

Train Samples      : 745702
Validation Samples : 159793
Test Samples       : 159794


In [6]:
feature_scaler = StandardScaler()

X_train_scaled = feature_scaler.fit_transform(X_train)

X_val_scaled = feature_scaler.transform(X_val)
X_test_scaled = feature_scaler.transform(X_test)

In [7]:
target_scaler = StandardScaler()

y_train_scaled = target_scaler.fit_transform(y_train.to_numpy().reshape(-1, 1))

y_val_scaled = target_scaler.transform(y_val.to_numpy().reshape(-1, 1))
y_test_scaled = target_scaler.transform(y_test.to_numpy().reshape(-1, 1))

In [8]:
# ============================================================
# QUBO-BASED LAG SELECTION
# ============================================================

LOOKBACK = 24
HORIZONS = [3, 12]
STRIDE = 1

NUM_LAGS = LOOKBACK
NUM_FEATURES = len(FEATURE_COLUMNS)

MI_SAMPLE_SIZE = 50_000
MI_RANDOM_STATE = 42
MI_N_NEIGHBORS = 3

ALPHA = 0.5
SA_RANDOM_STATE = 42

# QUBO lag numbering:
# lag 1 = most recent historical timestep
# lag 24 = oldest historical timestep

In [9]:
# For each forecast origin t:
#
# lag 1  = t-1
# lag 2  = t-2
# ...
# lag 24 = t-24
#
# Target = future Active_Power values immediately after t.

def build_mi_samples(
    X_data,
    y_data,
    lookback,
    horizon,
    sample_size,
    seed=42,
):
    X_data = np.asarray(X_data)
    y_data = np.asarray(y_data)

    valid_starts = np.arange(
        lookback,
        len(X_data) - horizon + 1
    )

    rng = np.random.default_rng(seed)

    if len(valid_starts) > sample_size:
        starts = rng.choice(
            valid_starts,
            size=sample_size,
            replace=False
        )
    else:
        starts = valid_starts

    starts = np.sort(starts)

    X_lagged = np.empty(
        (len(starts), lookback, X_data.shape[1]),
        dtype=np.float32
    )

    Y_future = np.empty(
        (len(starts), horizon),
        dtype=np.float32
    )

    for i, t in enumerate(starts):
        X_lagged[i] = X_data[t-lookback:t][::-1]
        Y_future[i] = y_data[t:t+horizon]

    return X_lagged, Y_future

In [10]:
# ============================================================
# COMPUTE RELEVANCE MI AND REDUNDANCY MI ONCE
# ============================================================

QUBO_HORIZON = 3

X_mi, Y_mi = build_mi_samples(
    X_train.to_numpy(),
    y_train.to_numpy(),
    LOOKBACK,
    QUBO_HORIZON,
    MI_SAMPLE_SIZE,
    MI_RANDOM_STATE,
)

# R_l:
# Average MI across all 7 features and 3 forecast horizons.

relevance_mi = np.zeros(NUM_LAGS)

for lag in range(NUM_LAGS):

    lag_position = lag

    values = []

    for feature_idx in range(NUM_FEATURES):

        x = X_mi[:, lag_position, feature_idx]

        for h in range(QUBO_HORIZON):

            mi = mutual_info_regression(
                x.reshape(-1, 1),
                Y_mi[:, h],
                n_neighbors=MI_N_NEIGHBORS,
                random_state=MI_RANDOM_STATE,
            )[0]

            values.append(mi)

    relevance_mi[lag] = np.mean(values)


# D_lm:
# Same-feature redundancy averaged over all 7 features.

redundancy_mi = np.zeros((NUM_LAGS, NUM_LAGS))

for l in range(NUM_LAGS):

    for m in range(l + 1, NUM_LAGS):

        values = []

        for feature_idx in range(NUM_FEATURES):

            x_l = X_mi[:, l, feature_idx]
            x_m = X_mi[:, m, feature_idx]

            mi = mutual_info_regression(
                x_l.reshape(-1, 1),
                x_m,
                n_neighbors=MI_N_NEIGHBORS,
                random_state=MI_RANDOM_STATE,
            )[0]

            values.append(mi)

        redundancy_mi[l, m] = np.mean(values)
        redundancy_mi[m, l] = redundancy_mi[l, m]

print("Relevance MI shape:", relevance_mi.shape)
print("Redundancy MI shape:", redundancy_mi.shape)

Relevance MI shape: (24,)
Redundancy MI shape: (24, 24)


In [11]:
def minmax_normalize(values):
    values = np.asarray(values, dtype=float)

    vmin = values.min()
    vmax = values.max()

    if np.isclose(vmax, vmin):
        return np.zeros_like(values)

    return (values - vmin) / (vmax - vmin)


R_norm = minmax_normalize(relevance_mi)

upper = redundancy_mi[np.triu_indices(NUM_LAGS, k=1)]
upper_norm = minmax_normalize(upper)

D_norm = np.zeros_like(redundancy_mi)

iu = np.triu_indices(NUM_LAGS, k=1)

D_norm[iu] = upper_norm
D_norm[(iu[1], iu[0])] = upper_norm

print("R range:", R_norm.min(), R_norm.max())
print("D range:", D_norm.min(), D_norm.max())

R range: 0.0 1.0
D range: 0.0 1.0


In [12]:
# ============================================================
# GREEDY RELEVANCE-REDUNDANCY CURVE
# No neural-network training is performed.
# Uses the already-computed MI matrices.
# ============================================================

selected = []
remaining = set(range(NUM_LAGS))

k_curve = []

for k in range(1, NUM_LAGS + 1):

    best_lag = None
    best_score = -np.inf

    for candidate in remaining:

        if selected:
            redundancy_penalty = np.mean(
                [D_norm[candidate, s] for s in selected]
            )
        else:
            redundancy_penalty = 0.0

        score = (
            ALPHA * R_norm[candidate]
            - (1 - ALPHA) * redundancy_penalty
        )

        if score > best_score:
            best_score = score
            best_lag = candidate

    selected.append(best_lag)
    remaining.remove(best_lag)

    k_curve.append({
        "K": k,
        "incremental_score": best_score,
        "selected_lags": [x + 1 for x in selected],
    })

k_curve_df = pd.DataFrame(k_curve)

display(k_curve_df)

,K,incremental_score,selected_lags
0,1,0.500000,[1]
1,2,0.071888,"[1, 8]"
2,3,0.095573,"[1, 8, 3]"
3,4,0.042134,"[1, 8, 3, 2]"
4,5,0.015392,"[1, 8, 3, 2, 5]"
5,6,0.003525,"[1, 8, 3, 2, 5, 14]"
6,7,0.019088,"[1, 8, 3, 2, 5, 14, 4]"
7,8,-0.013699,"[1, 8, 3, 2, 5, 14, 4, 6]"
8,9,-0.021122,"[1, 8, 3, 2, 5, 14, 4, 6, 11]"
9,10,-0.037907,"[1, 8, 3, 2, 5, 14, 4, 6, 11, 19]"


In [13]:
# ============================================================
# QUBO
# ============================================================

K = 12       # Set after inspecting k_curve_df
GAMMA = 1.0  # Only retained for the mathematical formulation

n = NUM_LAGS

Q = np.zeros((n, n), dtype=float)

# Linear relevance + cardinality terms
for l in range(n):

    Q[l, l] = (
        -ALPHA * R_norm[l]
        + GAMMA * (1 - 2 * K)
    )

# Redundancy + cardinality quadratic terms
for l in range(n):

    for m in range(l + 1, n):

        Q[l, m] = Q[m, l] = (
            ((1 - ALPHA) * D_norm[l, m] + 2 * GAMMA)
            / 2
        )

qubo_constant = GAMMA * K**2

In [14]:
def qubo_energy(x):
    return float(x @ Q @ x + qubo_constant)


def simulated_annealing(
    n,
    K,
    seed=42,
    restarts=8,
    sweeps=5000,
    cooling=0.995,
):
    rng = np.random.default_rng(seed)

    best_x = None
    best_energy = np.inf

    for _ in range(restarts):

        x = np.zeros(n, dtype=np.int8)

        x[rng.choice(n, size=K, replace=False)] = 1

        energy = qubo_energy(x)

        temperature = 1.0

        for _ in range(sweeps):

            selected = np.flatnonzero(x)
            unselected = np.flatnonzero(1 - x)

            remove_idx = rng.choice(selected)
            add_idx = rng.choice(unselected)

            new_x = x.copy()

            new_x[remove_idx] = 0
            new_x[add_idx] = 1

            new_energy = qubo_energy(new_x)

            delta = new_energy - energy

            if (
                delta <= 0
                or rng.random() < np.exp(
                    -delta / max(temperature, 1e-12)
                )
            ):
                x = new_x
                energy = new_energy

            if energy < best_energy:

                best_energy = energy
                best_x = x.copy()

            temperature *= cooling

    return best_x, best_energy


sa_x, sa_energy = simulated_annealing(
    n=NUM_LAGS,
    K=K,
    seed=SA_RANDOM_STATE,
)

sa_lags = (
    np.flatnonzero(sa_x) + 1
).tolist()

print("SA selected lags:", sa_lags)
print("SA energy:", sa_energy)
print("Number selected:", len(sa_lags))

SA selected lags: [1, 2, 3, 5, 7, 10, 13, 16, 19, 21, 23, 24]
SA energy: 8.897620960154057
Number selected: 12


In [15]:
# ============================================================
# EXACT FIXED-K VERIFICATION
# 24 choose 12 = 2,704,156 combinations
# ============================================================

best_exact_energy = np.inf
best_exact_lags = None

for combination in itertools.combinations(range(NUM_LAGS), K):

    x = np.zeros(NUM_LAGS, dtype=np.int8)
    x[list(combination)] = 1

    energy = qubo_energy(x)

    if energy < best_exact_energy:

        best_exact_energy = energy
        best_exact_lags = [
            i + 1 for i in combination
        ]

print("Exact selected lags:", best_exact_lags)
print("Exact energy:", best_exact_energy)

sa_matches_exact = bool(
    np.isclose(sa_energy, best_exact_energy)
    and sa_lags == best_exact_lags
)

print("SA matches exact:", sa_matches_exact)

Exact selected lags: [1, 2, 3, 5, 7, 10, 13, 16, 19, 21, 23, 24]
Exact energy: 8.897620960154057
SA matches exact: True


In [16]:
# ============================================================
# Selected-Lag Sequence Construction
# ============================================================

sa_lags = np.asarray(sa_lags, dtype=int)

def create_selected_lag_sequences(
    features,
    target,
    selected_lags,
    horizon,
    stride=1,
):
    X_seq = []
    y_seq = []

    selected_lags = np.asarray(selected_lags, dtype=int)

    last_start = len(features) - LOOKBACK - horizon + 1

    # Normalized temporal position for each selected lag
    lag_position = (
        selected_lags.astype(np.float32) / LOOKBACK
    ).reshape(-1, 1)

    for start in range(0, last_start, stride):

        end = start + LOOKBACK

        # lag 1 = most recent timestep
        # lag 24 = oldest timestep
        indices = end - selected_lags

        # Chronological order: oldest -> newest
        order = np.argsort(indices)
        indices = indices[order]

        # Corresponding lag positions must use the same order
        positions = lag_position[order]

        # 7 original features
        x_selected = features[indices]

        # Add temporal-position channel
        x_selected = np.concatenate(
            [x_selected, positions],
            axis=1
        )

        X_seq.append(x_selected)

        y_seq.append(
            target[end:end + horizon, 0]
        )

    return (
        torch.tensor(
            np.asarray(X_seq),
            dtype=torch.float32
        ),
        torch.tensor(
            np.asarray(y_seq),
            dtype=torch.float32
        ),
    )

In [17]:
# ============================================================
# Generate Selected-Lag Datasets
# ============================================================

datasets = {}

for horizon in HORIZONS:

    X_train_seq, y_train_seq = create_selected_lag_sequences(
        X_train_scaled,
        y_train_scaled,
        sa_lags,
        horizon,
        STRIDE,
    )

    X_val_seq, y_val_seq = create_selected_lag_sequences(
        X_val_scaled,
        y_val_scaled,
        sa_lags,
        horizon,
        STRIDE,
    )

    X_test_seq, y_test_seq = create_selected_lag_sequences(
        X_test_scaled,
        y_test_scaled,
        sa_lags,
        horizon,
        STRIDE,
    )

    datasets[horizon] = {
        "train": (X_train_seq, y_train_seq),
        "val": (X_val_seq, y_val_seq),
        "test": (X_test_seq, y_test_seq),
    }

    print(f"\n===== {horizon * 5} Minute Forecast =====")
    print("Selected lags:", sa_lags.tolist())
    print("Train :", X_train_seq.shape, y_train_seq.shape)
    print("Val   :", X_val_seq.shape, y_val_seq.shape)
    print("Test  :", X_test_seq.shape, y_test_seq.shape)


===== 15 Minute Forecast =====
Selected lags: [1, 2, 3, 5, 7, 10, 13, 16, 19, 21, 23, 24]
Train : torch.Size([745676, 12, 8]) torch.Size([745676, 3])
Val   : torch.Size([159767, 12, 8]) torch.Size([159767, 3])
Test  : torch.Size([159768, 12, 8]) torch.Size([159768, 3])



===== 60 Minute Forecast =====
Selected lags: [1, 2, 3, 5, 7, 10, 13, 16, 19, 21, 23, 24]
Train : torch.Size([745667, 12, 8]) torch.Size([745667, 12])
Val   : torch.Size([159758, 12, 8]) torch.Size([159758, 12])
Test  : torch.Size([159759, 12, 8]) torch.Size([159759, 12])


In [18]:
# ============================================================
# Inspect QUBO-Selected Lags
# ============================================================

print("QUBO-selected lags:")
print(sa_lags.tolist())

print("\nNumber of selected timestamps:", len(sa_lags))
print("Original lookback:", LOOKBACK, "timesteps")
print("Sampling interval: 5 minutes")

print("\nSelected timestamps:")
for lag in sorted(sa_lags):
    print(
        f"Lag {lag:2d} -> "
        f"{lag * 5:3d} minutes before forecast"
    )

QUBO-selected lags:
[1, 2, 3, 5, 7, 10, 13, 16, 19, 21, 23, 24]

Number of selected timestamps: 12
Original lookback: 24 timesteps
Sampling interval: 5 minutes

Selected timestamps:
Lag  1 ->   5 minutes before forecast
Lag  2 ->  10 minutes before forecast
Lag  3 ->  15 minutes before forecast
Lag  5 ->  25 minutes before forecast
Lag  7 ->  35 minutes before forecast
Lag 10 ->  50 minutes before forecast
Lag 13 ->  65 minutes before forecast
Lag 16 ->  80 minutes before forecast
Lag 19 ->  95 minutes before forecast
Lag 21 -> 105 minutes before forecast
Lag 23 -> 115 minutes before forecast
Lag 24 -> 120 minutes before forecast


In [19]:
print("Original features:")
for i, feature in enumerate(FEATURE_COLUMNS, 1):
    print(f"{i}. {feature}")

print("\nFinal input channels:")
for i, feature in enumerate(FEATURE_COLUMNS, 1):
    print(f"{i}. {feature}")

print(f"{len(FEATURE_COLUMNS)} original features + 1 Lag_Position = 8 channels")

Original features:
1. Weather_Temperature_Celsius
2. Weather_Relative_Humidity
3. Global_Horizontal_Radiation
4. Diffuse_Horizontal_Radiation
5. Radiation_Global_Tilted
6. Radiation_Diffuse_Tilted
7. Active_Power

Final input channels:
1. Weather_Temperature_Celsius
2. Weather_Relative_Humidity
3. Global_Horizontal_Radiation
4. Diffuse_Horizontal_Radiation
5. Radiation_Global_Tilted
6. Radiation_Diffuse_Tilted
7. Active_Power
7 original features + 1 Lag_Position = 8 channels


In [20]:
sample_X = datasets[3]["train"][0][0]

print("Sample shape:", sample_X.shape)
print("\nSelected sequence:")

actual_lags_in_sequence = sorted(sa_lags, reverse=True)

for i, lag in enumerate(actual_lags_in_sequence):
    print(
        f"Position {i:2d} | "
        f"Lag {lag:2d} | "
        f"{lag * 5:3d} min ago | "
        f"Lag_Position = {sample_X[i, -1].item():.4f}"
    )

Sample shape: torch.Size([12, 8])

Selected sequence:
Position  0 | Lag 24 | 120 min ago | Lag_Position = 1.0000
Position  1 | Lag 23 | 115 min ago | Lag_Position = 0.9583
Position  2 | Lag 21 | 105 min ago | Lag_Position = 0.8750
Position  3 | Lag 19 |  95 min ago | Lag_Position = 0.7917
Position  4 | Lag 16 |  80 min ago | Lag_Position = 0.6667
Position  5 | Lag 13 |  65 min ago | Lag_Position = 0.5417
Position  6 | Lag 10 |  50 min ago | Lag_Position = 0.4167
Position  7 | Lag  7 |  35 min ago | Lag_Position = 0.2917
Position  8 | Lag  5 |  25 min ago | Lag_Position = 0.2083
Position  9 | Lag  3 |  15 min ago | Lag_Position = 0.1250
Position 10 | Lag  2 |  10 min ago | Lag_Position = 0.0833
Position 11 | Lag  1 |   5 min ago | Lag_Position = 0.0417


In [21]:
QUBO_ARTIFACT_DIR = PROJECT_ROOT / "artifacts" / "qubo_lag_selection"
QUBO_ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print("QUBO artifact directory:", QUBO_ARTIFACT_DIR)

QUBO artifact directory: C:\Users\silsw\OneDrive\Desktop\PV-forecasting\artifacts\qubo_lag_selection


In [22]:
# ============================================================
# Save QUBO Preprocessing Artifacts
# ============================================================

with open(QUBO_ARTIFACT_DIR / "feature_scaler.pkl", "wb") as f:
    pickle.dump(feature_scaler, f)

with open(QUBO_ARTIFACT_DIR / "target_scaler.pkl", "wb") as f:
    pickle.dump(target_scaler, f)


qubo_config = {
    "lookback": LOOKBACK,

    "selected_lags": sa_lags.tolist(),
    "num_selected_lags": len(sa_lags),

    "sampling_interval_minutes": 5,

    "horizons": HORIZONS,
    "forecast_minutes": {
        str(h): h * 5 for h in HORIZONS
    },

    "active_horizon": 3,
    "active_forecast_minutes": 15,

    "stride": STRIDE,

    "original_features": FEATURE_COLUMNS,
    "num_original_features": len(FEATURE_COLUMNS),

    "additional_input_channel": "Lag_Position",
    "final_num_input_channels": len(FEATURE_COLUMNS) + 1,

    "target": TARGET_COLUMN,

    "train_ratio": TRAIN_RATIO,
    "val_ratio": VAL_RATIO,
    "test_ratio": TEST_RATIO,

    "feature_scaler": "StandardScaler",
    "target_scaler": "StandardScaler",

    "qubo": {
        "alpha": ALPHA,
        "gamma": GAMMA,
        "num_variables": NUM_LAGS,
        "solver": "Simulated Annealing",
        "exact_verification": True,
        "sa_matches_exact": bool(sa_matches_exact),
    },
}


with open(
    QUBO_ARTIFACT_DIR / "preprocessing_config.json",
    "w"
) as f:
    json.dump(qubo_config, f, indent=4)


print("QUBO preprocessing artifacts saved successfully.\n")

for file in sorted(QUBO_ARTIFACT_DIR.iterdir()):
    print(file.name)

QUBO preprocessing artifacts saved successfully.



feature_scaler.pkl
preprocessing_config.json
target_scaler.pkl


In [23]:
np.save(
    QUBO_ARTIFACT_DIR / "selected_lags.npy",
    sa_lags
)

with open(
    QUBO_ARTIFACT_DIR / "selected_lags.json",
    "w"
) as f:
    json.dump(
        {
            "selected_lags": sa_lags.tolist(),
            "selected_minutes_before_forecast": (
                (sa_lags * 5).tolist()
            ),
        },
        f,
        indent=4,
    )

In [24]:
# ============================================================
# Save QUBO Selected-Lag Datasets
# ============================================================

for horizon in HORIZONS:

    forecast_minutes = horizon * 5

    torch.save(
        {
            "X": datasets[horizon]["train"][0],
            "y": datasets[horizon]["train"][1],
        },
        QUBO_ARTIFACT_DIR / f"train_{forecast_minutes}.pt",
    )

    torch.save(
        {
            "X": datasets[horizon]["val"][0],
            "y": datasets[horizon]["val"][1],
        },
        QUBO_ARTIFACT_DIR / f"val_{forecast_minutes}.pt",
    )

    torch.save(
        {
            "X": datasets[horizon]["test"][0],
            "y": datasets[horizon]["test"][1],
        },
        QUBO_ARTIFACT_DIR / f"test_{forecast_minutes}.pt",
    )

    print(f"Saved {forecast_minutes}-minute QUBO datasets.")

Saved 15-minute QUBO datasets.


Saved 60-minute QUBO datasets.
